# ポートリャーギンの最小原理(PMP)の導出

変分法との対比


|項目|変分法|PMP|
|:---|:---|:---|
|評価関数 |$J[y]$ | $J[x,u]$|
|最適化する対象|関数 $y(x)$ | 状態軌道 $x(t)$と入力軌道 $u(t)$ |
|制約条件|なし(基本形)|状態方程式 $\dot{x}=f(x,u)$|
|必要条件|オイラー・ラグランジュ方程式|状態方程式・随伴方程式・最適性条件|
|終端の考慮|部分積分による境界項|部分積分による境界項|
|考慮する変分|$\delta y = \varepsilon \eta(x)$ | $\delta x, \delta u, \delta \lambda$|

変分法では、$J[y]$を最小化する問題だったが、<br>
PMPでは $\dot{x}=f(x,u)$という制約が追加される。この制約を評価関数に組み込むために、ラグランジュ未定乗数 $\lambda(t)$を導入する。<br>
すると元の評価関数$J$にラグランジュ項を追加した「拡張評価関数」($\bar{J}$)を作ることになる。
$$
\bar{J}[x,u,\lambda] = \Phi(x(T)) + \int_0^T (L + \lambda^T(f-\dot{x}))dt
$$

変分法は「関数$y(x)$の変分 $\delta y$を考え、評価関数が停留する条件を求める」というものであり、<br>
PMPは「状態軌道$x(t)$と入力軌道 $u(t)$ に加え、状態方程式を制約として組み込むためのラグランジュ未定乗数 $\lambda(t)$を導入する。そして$x,u,\lambda$のすべての変分を考え、評価関数が停留する条件を求める」というものである。<br>

PMPは変分法とラグランジュ未定乗数法の考え方を用いて、状態方程式という制約付き最適化問題に対する必要条件を与えるものである。

## 順番

PMPの導出まで長い道のりであるため、以下の流れを示す

1. 最適制御問題の記述
1. 状態方程式を評価関数へ組み込む
1. 拡大評価関数の変分計算
1. 部分積分による変分の消去
1. PMP条件の抽出
1. HamiltonianによるPMP条件記述の簡略化


## 最適制御問題

最小化したい評価関数 $J[x,u]$を次とする。

$$
J[x,u] = \Phi(x(T)) + \int_0^T L(x(t), u(t), t) dt
$$

ここで
- $x(t)$ : 状態(state)
- $u(t)$ : 制御入力(control input)
- $T$ : 終端時刻

$J[x,u]$の各項目の意味を以下に説明する。

### 終端コスト $\Phi(x(T))$

評価関数を評価する期間の最終時刻$T$での状態(state)を評価する。<br>
例えば次のような状態
- 倒立揺子を真上で安定化させたい(角度=0, 角速度=0)
- ロボットの手先を目標位置姿勢にしたい ($x=X_T, y=Y_T, z=Z_T, \alpha = \alpha_T, \beta=\beta_T, \gamma=\gamma_T$)

具体的には、倒立揺子の状態を $x = [\theta, \omega]$ とし、最終時刻での状態を $x(T) = [\theta(T), \omega(T)]$、そこでの目標状態を$x_{ref} =  [\theta_T, \omega_T]$とするなら、

$$
\Phi(x(T)) = (x(T) - x_{ref})^T \mathbf{Q}_T (x(T) - x_{ref}) = (\theta(T) - \theta_{T}, \omega(T) - \omega_{T})^T \mathbf{Q}_T (\theta(T) - \theta_{T}, \omega(T) - \omega_{T})
$$
となり、これは最終状態がどれくらい目標状態に近いかを評価する。

$\mathbf{Q}_T$は通常、対称な正定値(または半正定値)行列であり、各状態の重みを表す。<br>
例えば以下のような行列である。
$$
\begin{bmatrix} 10 & 0 \\ 0 & 50 \end{bmatrix}, \space
\begin{bmatrix} 10 & 0 \\ 0 & 0 \end{bmatrix}
$$

2次形式であるため、スカラーであることに注意する。

### ランニングコスト $\int_0^T L(x,u,t) dt$

$\int_0^T L(x,u,t) dt$は区間[0,T]まで終端時刻に至るまでの状態$x$と制御入力$u$を評価する。
例えば、
$$L=x^T Q x + u^T R u$$
である。これは、「状態誤差を小さくしたい」、「制御入力をつかいすぎないようにしたい」という目的を示している。

また、目標状態 $x_{ref}$ を考慮するなら、以下の $L$ となり、この場合は 状態誤差($x-x_{ref}$)を小さくしたいという目的にになる。
$$
L=(x - x_{ref})^T Q (x- x_{ref}) + u^T R u
$$

そして2次形式で表しているため、$L$はスカラーであることに注意する。

### 制約の考慮

状態$x$は物理法則に従って変化するため、自由に選ぶことはできない。例えば時刻 k のとき 位置:0, 速度:0で、時刻 k+1 で位置:0, 速度:100などは物理法則に反している。<br>
例えばロボットでは、質量・慣性・重力などを考慮した運動方程式に従って運動するため、状態は入力による決まるため任意には決められない。<br>

倒立揺子である場合、次のような状態方程式になる。
$$
\dot{x} = \begin{bmatrix} \dot{p} \\ \ddot{p} \\ \dot{\theta} \\ \ddot{\theta} \end{bmatrix} = f(x,u)
$$

また、初期状態として、$x(0) = x_0$ という初期条件があり、これはスタート地点が固定されていることを意味してる。

PMPは次の最適制御問題を解くための条件である。

$$
\min\limits_{x(t), u(t)} \space J = \Phi(x(T)) + \int_0^T L (x,u,t) dt \\
\text{subject to} \quad \dot{x} = f(x,u,t), \space x(0) = x_0
$$

これは、「初期状態を$x_0$とする状態方程式に従って動く状態軌道 $x(t)$ と入力軌道 $u(t)$ の中から、評価関数 $J$ を最小にするものを求める」という問題である。

上記は数学的な見方として$x(t), u(t)$ を未知関数として扱っているが、状態方程式と初期条件が与えられれば、入力軌道 $u(t)$ を決めることで状態軌道 $x(u)$ が一意に決まる。そのため工学では、「最適な入力軌道$u(t)$を求める問題」と説明されることがある。

このノートでは、数学的な導出に従い、状態軌道$x(t)$と入力軌道$u(t)$の両方を未知関数として扱う。

ここまでは、最適制御問題の定義である。次に、この状態方程式という制約を評価関数へ組み込むため、ラグランジュ未定乗数を導入し、拡張評価関数を定義する。

## 状態方程式を評価関数へ組み込む

変分法では評価関数$J[y]$を停留させる関数を$\delta J=0$ の条件から求めていた。この時は制約条件なしであった。

最適制御問題では状態方程式 $\dot{x} = f(x,u,t)$ が制約となるため、これを評価関数へラグランジュの未定乗数法のように組み込む必要がある。

ラグランジュの未定乗数法では、$\min_x f(x) \quad \text{subject to} \quad g(x) = 0$であり、$\bar{f}(x,\lambda) = f(x) + \lambda g(x)$ という拡張目的関数を作った。ラグランジュの未定乗数法は 制約条件 $g(x)=0$ をラグランジュ未定乗数$\lambda$によって評価関数へ組み込み、制約付き最適化問題を、拡張目的関数の停留条件を求める問題へ書き換える。<br>

PMPでは制約である状態方程式 $\dot{x} = f(x,u,t)$ を $f(x,u,t) - \dot{x} = 0 $とすると、これは ラグランジュの未定乗数法における $g(x) = 0$ と同様であるため、ラグランジュ未定乗数 $\lambda(t)$ を制約条件 $f(x,u,t) - \dot{x} = 0 $ にかけ、その結果を評価関数へ加える。

状態方程式は一般に複数の状態変数から構成される。つまり状態 $x$は
$$
x = \begin{bmatrix} x_1 \\ x_2 \\ \vdots \\ x_n \end{bmatrix}
$$
となるため、 $f(x,u,t) - \dot{x}$も同じ次元の縦ベクトルになる。
ラグランジュ未定乗数$\lambda(t)$ も

$$
\lambda = \begin{bmatrix} \lambda_1 \\ \lambda_2 \\ \vdots \\ \lambda_n \end{bmatrix}
$$

であるため、$\lambda^T(f(x,u,t)-\dot{x})$ を評価関数へ組み込む。<br>
状態方程式は評価期間$[0,T]$のすべてで満たす必要がある。そのため一点だけを評価する終端コストではなく、ランニングコストである時間全体を評価する積分項へ組み込み、次の拡張評価関数 $\bar{J}$を定義する。

$$
\bar{J}[x,u,\lambda] = \Phi(x(T)) + \int_0^T \left( L(x,u,t) + \lambda^T (f(x,u,t) - \dot{x})\right) dt
$$

状態方程式を満たす軌道では$(f(x,u,t)-\dot{x}) = 0$となるため、目的は最初と同じ $J = \Phi + \int L dt$ を最小化することと同一である。<br>
これは、状態方程式を満たす軌道に対しては $\bar{J} = J$ であるということになる。 
したがって、拡張評価関数$\bar{J}$は元の目的関数を変更したものではなく、状態方程式という制約を評価関数へ組み込むために導入した関数である。

この拡張評価関数に対して変分を取り、停留条件を求めることで、PMPの基本方程式を導出する。

## 拡大評価関数の変分計算

変分法では$y(x) + \varepsilon \eta(x)$を考えたのと同様に、PMPでは状態 $x$、入力 $u$、ラグランジュ未定乗数 $\lambda$をそれぞれ以下のように微小変化させる。
$$
x + \varepsilon \delta x , \space u + \varepsilon \delta u, \space \lambda + \varepsilon \delta \lambda
$$

----------------------------------------------------------
ここで次の補足を加える。

変分法では変分$\varepsilon \eta(x)$による近傍の変化量を以下のように記述した。
$$
J[y+\varepsilon \eta(x)] = J[y] + \delta J \varepsilon + O(\varepsilon^2)
$$
ここからJの一次変化 $\delta J$は次のようにも導出できる。

$$
\begin{array}{ll}
\delta J & = \left. \dfrac{d}{d \varepsilon} J[y+\varepsilon \eta(x)] \ \right|_{\varepsilon=0} \\
& = \left. \dfrac{d}{d \varepsilon} \left(  J[y] + \delta J \varepsilon + O(\varepsilon^2) \right) \ \right|_{\varepsilon=0} \\
& = \left. \left( 0 + \delta J + O(\varepsilon) \right) \ \right|_{\varepsilon=0} \\
& = \delta J
\end{array}
$$

$J[y]$は定数であり、$O(\varepsilon^2)=a\varepsilon^2 + b\varepsilon^3 \cdots$のような関数である。なお  $O(\varepsilon^2)$には定数は入らない。

つまり、評価関数 $J[y + \varepsilon \eta]$ に関して、$\varepsilon$へ微分して、$\varepsilon=0$における一次変化を求めたものが$\delta J$である。このように、「関数へ微小な変化を与え、その一次変化を求める操作」を「変分を取る」と呼ぶ。

そして、$\delta$という記号を以下の操作を行うこととして定義する。

$$
\delta = \left. \frac{d}{d \varepsilon} \right|_{\varepsilon=0}
$$

--------------------------------------------------------------

#### Step1 : 拡大評価関数

拡大評価関数 $\bar{J}$は以下である。

$$
\bar{J}[x,u,\lambda] = \Phi(x(T)) + \int_0^T \left( L(x,u,t) + \lambda^T (f(x,u,t) - \dot{x})\right) dt
$$

#### Step2 : 微小変化を代入した拡大評価関数

状態 $x$、入力 $u$、ラグランジュ未定乗数 $\lambda$をそれぞれ以下のように微小変化させる。
$$
\begin{array}{l}
x \rightarrow x + \varepsilon \delta x \\
u \rightarrow u + \varepsilon \delta u \\
\lambda \rightarrow  \lambda + \varepsilon \delta \lambda
\end{array}
$$

終端状態$x(T)$は以下となる。
$$
x(T) \rightarrow x(T) + \varepsilon \delta x(T)
$$

速度項 $\dot{x}$は以下となる。
$$
\dot{x} \rightarrow \frac{d}{dt}\left(x + \varepsilon \delta x \right) \rightarrow \dot{x} + \varepsilon \delta \dot{x}
$$

したがって、微小変化を代入した拡張評価関数 $\bar{J}$は以下となる。

$$
\begin{aligned}
&\bar{J}
\left[
x+\varepsilon\delta x,\,
u+\varepsilon\delta u,\,
\lambda+\varepsilon\delta\lambda
\right]
\\
&=
\Phi
\left(
x(T)+\varepsilon\delta x(T)
\right)
\\
&\quad+
\int_0^T
\Bigg[
L
\left(
x+\varepsilon\delta x,\,
u+\varepsilon\delta u,\,
t
\right)
\\
&\qquad+
\left(
\lambda+\varepsilon\delta\lambda
\right)^T
\Bigg(
f
\left(
x+\varepsilon\delta x,\,
u+\varepsilon\delta u,\,
t
\right)
-
\left(
\dot{x}+\varepsilon\delta\dot{x}
\right)
\Bigg)
\Bigg]dt
\end{aligned}
$$


#### Step 3 : 変分 $\delta$ の適用

変分 $\delta$ とは以下の操作を行うものであった。
$$
\delta = \left. \frac{d}{d \varepsilon} \right|_{\varepsilon=0}
$$ 

Step 2で求めた微小変化を代入した拡張評価関数 $\bar{J}$の変分の計算を行う。<br>
左辺は以下となる。

$$
\begin{aligned}
\delta\bar{J}
&=
\left.
\frac{d}{d\varepsilon}
\bar{J}
\left[
x+\varepsilon\delta x,\,
u+\varepsilon\delta u,\,
\lambda+\varepsilon\delta\lambda
\right]
\right|_{\varepsilon=0}
\end{aligned}
$$

右辺は以下のようになる。

$$
\begin{aligned}
\delta\bar{J}&= \frac{d}{d\varepsilon} \Bigg[ \Phi \left(x(T)+\varepsilon\delta x(T) \right) \\
&\qquad+\int_0^T \Bigg( L \left( x+\varepsilon\delta x, \ u+\varepsilon\delta u, \ t \right) \\
&\qquad+\left( \lambda+\varepsilon\delta\lambda \right)^T \Bigg( f \left( x+\varepsilon\delta x, \  u+\varepsilon\delta u, \  t \right)-\left( \dot{x}+\varepsilon\delta\dot{x} \right) \Bigg) \Bigg) dt \Bigg]
\Bigg|_{\varepsilon=0}
\end{aligned}
$$

#### Step 4 項目の分割

上式は以下のように表現することができる。

$$
\delta \bar{J} = \left. \frac{d}{d \varepsilon} \left[ \Phi(x(T) + \varepsilon \delta x(T)) + \int_0^T A(\varepsilon, t)  dt \right] \right|_{\varepsilon = 0}
$$

ここで、$A(\varepsilon, t)$は以下である。
$$
\begin{aligned}
A(\varepsilon, t) =& L \left( x+\varepsilon\delta x, \ u+\varepsilon\delta u, \ t \right) \\
&\qquad+\left( \lambda+\varepsilon\delta\lambda \right)^T \left( f \left( x+\varepsilon\delta x, \  u+\varepsilon\delta u, \  t \right)-\left( \dot{x}+\varepsilon\delta\dot{x} \right) \right)
\end{aligned}
$$

$\delta \bar{J}$を項目ごとに分けると、次のようになる。

$$
\begin{aligned}
\delta \bar{J} &= \left. \frac{d}{d \varepsilon} \Phi(x(T) + \varepsilon \delta x(T)) \right|_{\varepsilon=0} \\ &\quad + \frac{d}{d \varepsilon} \left. \int_0^T A(\varepsilon, t)  dt  \right|_{\varepsilon = 0}
\end{aligned}
$$

積分は$t$に関して行われる。$\varepsilon$は$t$に関しては単なるパラメータであるので、積分後も$\varepsilon$の関数として$A(\varepsilon)$は残る。そのため、その後に$\varepsilon$ で微分できる。また、積分区間[0,T]が$\varepsilon$に依存しないため、微分と積分の順序を交換できる。よって、$\delta \bar{J}$は次のようになる。

$$
\begin{aligned}
\delta \bar{J} &= \left. \frac{d}{d \varepsilon} \Phi(x(T) + \varepsilon \delta x(T)) \right|_{\varepsilon=0} \\ &\quad +  \left. \int_0^T \frac{d}{d \varepsilon} A(\varepsilon, t)  \right|_{\varepsilon = 0} dt
 \end{aligned}
$$

-------------------------------------------------------
上記の微分と積分の順序を入れ替えることが出来ることを$A(\varepsilon,t) = \varepsilon t + t^2$ と簡単な例に置き換えて確認する。

$$
\int_0^T A(\varepsilon,t) dt = \int_0^T (\varepsilon t + t^2) dt = \varepsilon \frac{T^2}{2} + \frac{T^3}{3}
$$
この結果から分かるように、$\varepsilon$ は$t$のパラメータであり、積分後は変数$t$が定数$T$となるため$A(\varepsilon)$の関数が残る。

これを $\varepsilon$で微分すると、$T^2/2$となる。
$$
\frac{d}{d \varepsilon}\left( \varepsilon \frac{T^2}{2} + \frac{T^3}{3} \right) = \frac{T^2}{2}
$$

一方、積分の中で先に微分して、その後その結果を積分すると、同じく$T^2/2$となる。

$$
\frac{\partial A(\varepsilon, t)}{\partial \varepsilon} = t \rightarrow \int_0^T t dt = \frac{T^2}{2}
$$ 

---------------------------------------------------------

#### Step 5 : 終端コスト $\Phi$ の変分

Step 4 で導出した$\delta \bar{J}$式の第1項の変分計算を行う。

$$
\delta \Phi = \left. \frac{d}{d \varepsilon} \Phi(x(T) + \varepsilon \delta x(T)) \right|_{\varepsilon=0}
$$

チェインルールを確認する。

例えば $y = f(g(\varepsilon))$なら、以下のようになる。
$$
\frac{d \ y}{d \ \varepsilon} = \frac{d \ f}{d \ g} \frac{d \ g}{d \ \varepsilon}
$$

今回の $\delta \Phi$ の計算では、$g = x(T) + \varepsilon \delta x(T)$ となるので、

$$
\frac{d}{d \varepsilon} \Phi(x(T) + \varepsilon \delta x(T)) = \frac{\partial \Phi(x(T)+\varepsilon \delta x(T))}{\partial (x(T) + \varepsilon \delta x(T))} \frac{d}{d \varepsilon}(x(T) + \varepsilon \delta x(T)) = \frac{\partial \Phi(x(T)+\varepsilon \delta x(T))}{\partial (x(T) + \varepsilon \delta x(T))} \delta x(T)  
$$

最後に $\varepsilon=0$ として$\delta \Phi$を求める。
$$
\delta \Phi = \left(\frac{\partial \Phi(x(T))}{\partial x(T)} \right)^T \delta x(T) = \Phi_x^T \delta x(T)
$$

拡張評価関数$\bar{J}[x,u, \lambda]$はスカラーであり、その一次変化の第一項である $\delta \Phi$もスカラーである必要がある。<br>
ここでは、$\Phi_x, \delta x(T)$を縦ベクトルとしているため、スカラー計算のため$\Phi_x^T$としている。

$$
\Phi_x = \frac{\partial \Phi(x(T))}{\partial x(T)} = \begin{bmatrix}
\partial \Phi(x(T)) / \partial x_1(T) \\
\vdots \\
\partial \Phi(x(T)) / \partial x_n(T) \\
\end{bmatrix}
$$



#### Step 6 ランニングコスト $A(\varepsilon, t)$ の変分

Step 4 で導出した$\delta \bar{J}$式の第2項の変分計算を行う。


$$
\left. \int_0^T \frac{d}{d \varepsilon}  A(\varepsilon, t)  \right|_{\varepsilon = 0} dt
$$

ここで、$A(\varepsilon, t)$は以下である。
$$
\begin{aligned}
A(\varepsilon, t) =& L \left( x+\varepsilon\delta x, \ u+\varepsilon\delta u, \ t \right) \\
&\qquad+\left( \lambda+\varepsilon\delta\lambda \right)^T \left( f \left( x+\varepsilon\delta x, \  u+\varepsilon\delta u, \  t \right)-\left( \dot{x}+\varepsilon\delta\dot{x} \right) \right)
\end{aligned}
$$


------------------------------------------------------------------
具体的な変分計算の前に式の見通しをよくするために、$L, f$に関する$t$に関する表示を省略する。

この理由は、例えば $G(x(t)+\varepsilon \delta x(t) ,t)$を $\varepsilon$で微分しその後 $\varepsilon=0$ とする変分計算を行うとすると、
$$
\left. \frac{d}{d \varepsilon}\left(G(x(t)+\varepsilon \delta x(t),t)\right) \right|_{\varepsilon=0}= \left. \left( \frac{\partial G(x(t)+\varepsilon \delta x(t),t)}{\partial (x(t) + \varepsilon \delta x(t))}\frac{d}{d \ \varepsilon}(x(t)+\varepsilon \delta x(t)) + \frac{\partial G(x(t)+\varepsilon \delta x(t),t)}{\partial t}\frac{d \ t}{d \ \varepsilon} \right) \right|_{\varepsilon=0}
$$

ここで、$dt / d \varepsilon = 0$ となり、$\varepsilon=0$ とすると、以下となる。

$$
\left. \frac{d}{d \varepsilon}\left(G(x(t)+\varepsilon \delta x(t),t)\right) \right|_{\varepsilon=0} = \frac{\partial G(x(t),t)}{\partial (x(t))} \delta x
$$

$G(x(t), t)$を$x(t)$で偏微分する場合、$t$は$x(t)$に対して固定とみなせる。そのため$x(t)$の偏微分では、"t"自身や"t"を含む部分が変形されることはない。これは例えば、$at x^2 = (at) x^2$を$x$で偏微分した時 $(at) 2 x$ となって、"t"を含む部分 $at$ はそのまま残る。このように、$x(t)$による偏微分では$t$を含む部分は変化せず式中に残り続ける。"t"が式に残り続けることを理解した上で、記述を簡潔にするため第2引数の"t"を省略する。

$$
\left. \frac{d}{d \varepsilon}\left(G(x(t)+\varepsilon \delta x(t),t)\right) \right|_{\varepsilon=0} = \frac{\partial G(x(t))}{\partial x(t)} \delta x = G_x \delta x
$$

注意として、$t$を省略して記述しているだけで、依然として$t$の関数であることに変わりはない。

また、$x(t), u(t), \lambda(t)$ は 時間 "t" の関数であるが、各時刻$t$における値を変数として扱うため、記述を簡潔にするために以下のように省略して表記する。

$$
x(t), u(t), \lambda(t) \rightarrow x, u, \lambda
$$

------------------------------------------------------------------


第2項の変分計算を$t$を省略した状態で行っていく。
$$
\left. \frac{d}{d \varepsilon}  A(\varepsilon, t)  \right|_{\varepsilon = 0}
$$

$A(\varepsilon, t)$は次のものである。
$$
\begin{aligned}
A(\varepsilon, t) =& L \left( x+\varepsilon\delta x, \ u+\varepsilon\delta u \right) \\
&\qquad+\left( \lambda+\varepsilon\delta\lambda \right)^T \left( f \left( x+\varepsilon\delta x, \  u+\varepsilon\delta u \right)-\left( \dot{x}+\varepsilon\delta\dot{x} \right) \right)
\end{aligned}
$$


##### L の項 の変分計算

$$
\delta L = \left. \frac{d}{d \varepsilon}L \left( x+\varepsilon\delta x, \ u+\varepsilon\delta u \right) \right|_{\varepsilon=0}
$$

チェインルールより$\varepsilon$による微分は以下となる。
$$
\begin{aligned}
=& \left[ \frac{\partial L \left( x+\varepsilon\delta x, \ u+\varepsilon\delta u \right) }{\partial (x+\varepsilon\delta x)} \right]^T \frac{d}{d \varepsilon} (x+\varepsilon\delta x) + \left[ \frac{\partial L \left( x+\varepsilon\delta x, \ u+\varepsilon\delta u \right) }{\partial (u+\varepsilon\delta u)} \right]^T \frac{d}{d \varepsilon} (u+\varepsilon\delta u) \\
=& \left[ \frac{\partial L \left( x+\varepsilon\delta x, \ u+\varepsilon\delta u \right) }{\partial (x+\varepsilon\delta x)} \right]^T \delta x + \left[ \frac{\partial L \left( x+\varepsilon\delta x, \ u+\varepsilon\delta u \right) }{\partial (u+\varepsilon\delta u)} \right]^T \delta u
\end{aligned}
$$

次に $\varepsilon=0$ を代入する。
$$
\begin{aligned}
=& \left[ \frac{\partial L \left( x, \ u \right) }{\partial x} \right]^T \delta x + \left[ \frac{\partial L \left( x, \ u\right) }{\partial u} \right]^T \delta u \\
=& {L_x}^T \delta x + {L_u}^T \delta u
\end{aligned}
$$

よって、$L$に関する変分計算は次になる。
$$
\left. \frac{d}{d \varepsilon}L \left( x+\varepsilon\delta x, \ u+\varepsilon\delta u \right) \right|_{\varepsilon=0} = {L_x}^T \delta x + {L_u}^T \delta u
$$


##### $\lambda^T f$ の項の変分計算

$$
\delta \left( \lambda^T f \right) = \left. \frac{d}{d \varepsilon} \left[
\left( \lambda+\varepsilon\delta\lambda \right)^T \left( f \left( x+\varepsilon\delta x, \  u+\varepsilon\delta u \right) \right) \right] \right|_{\varepsilon=0}
$$

積の微分より

$$
\begin{aligned}
& \frac{d}{d \varepsilon} \left[
\left( \lambda+\varepsilon\delta\lambda \right)^T \left( f \left( x+\varepsilon\delta x, \  u+\varepsilon\delta u \right) \right) \right] \\
= & \left( \frac{d}{d \varepsilon} \left( \lambda+\varepsilon\delta\lambda \right)^T \right) f \left( x+\varepsilon\delta x, \  u+\varepsilon\delta u \right) + \left( \lambda+\varepsilon\delta\lambda \right)^T \left(\frac{d}{d \varepsilon} f \left( x+\varepsilon\delta x, \  u+\varepsilon\delta u \right)\right)
\end{aligned}
$$

それぞれの $\varepsilon$ による微分を計算する。

1項目

$$
\frac{d}{d \varepsilon} \left( \lambda+ \varepsilon \delta\lambda \right)^T = \delta\lambda ^T 
$$

2項目

$$
\begin{aligned}
&\frac{d}{d \varepsilon} f \left( x+\varepsilon\delta x, \  u+\varepsilon\delta u \right) \\
=& \frac{\partial f \left( x+\varepsilon\delta x, \  u+\varepsilon\delta u \right)}{\partial (x+\varepsilon\delta x)} \frac{d}{d \varepsilon}(x+\varepsilon\delta x) + \frac{\partial f \left( x+\varepsilon\delta x, \  u+\varepsilon\delta u \right)}{\partial (u+\varepsilon\delta u)} \frac{d}{d \varepsilon}(u+\varepsilon\delta u) \\
=& \frac{\partial f \left( x+\varepsilon\delta x, \  u+\varepsilon\delta u \right)}{\partial (x+\varepsilon\delta x)} \delta x + \frac{\partial f \left( x+\varepsilon\delta x, \  u+\varepsilon\delta u \right)}{\partial (u+\varepsilon\delta u)} \delta u
\end{aligned}
$$

これらを積の微分の式に代入すると、

$$
\begin{aligned}
&\left( \frac{d}{d \varepsilon} \left( \lambda+\varepsilon\delta\lambda \right)^T \right) f \left( x+\varepsilon\delta x, \  u+\varepsilon\delta u \right) + \left( \lambda+\varepsilon\delta\lambda \right)^T \left(\frac{d}{d \varepsilon} f \left( x+\varepsilon\delta x, \  u+\varepsilon\delta u \right)\right) \\
=& \delta\lambda ^T  f \left( x+\varepsilon\delta x, \  u+\varepsilon\delta u \right) + \left( \lambda+\varepsilon\delta\lambda \right)^T \left(\frac{\partial f \left( x+\varepsilon\delta x, \  u+\varepsilon\delta u \right)}{\partial (x+\varepsilon\delta x)} \delta x + \frac{\partial f \left( x+\varepsilon\delta x, \  u+\varepsilon\delta u \right)}{\partial (u+\varepsilon\delta u)} \delta u \right)
\end{aligned}
$$

さらに$\varepsilon=0$を代入すると、


$$
\begin{aligned}
\delta\lambda ^T  f \left( x, \  u \right) + \lambda ^T \left(\frac{\partial f \left( x, \  u\right)}{\partial x} \delta x + \frac{\partial f \left( x , \  u \right)}{\partial u}  \delta u \right) \\ 
\end{aligned}
$$

$f(x,u)$は 運動方程式 $\dot{x} = f$ のことである。また、表記の簡略化のため $f_x = \partial f / \partial x$ , $f_u = \partial f / \partial u$ として、

$$
\begin{aligned}
\delta \left( \lambda^T f \right) = \delta \lambda ^T  f  + \lambda ^T f_x  \delta x + \lambda ^T f_u \delta u \\
\end{aligned}
$$

後で$\delta x, \delta u$ の係数を整理するため、ベクトルの向きをそろえる。

$$
\begin{aligned}
\lambda ^T f_x  \delta x = \left(f_x ^T \lambda \right)^T \delta x\\ 
\lambda ^T f_u \delta u = \left(f_u ^T \lambda \right)^T \delta u 
\end{aligned}
$$

より

$$
\begin{aligned}
\delta \left( \lambda^T f \right) = \delta \lambda ^T  f  + ( f_x ^T \lambda) ^T \delta x + (f_u ^T \lambda) ^T \delta u \\
\end{aligned}
$$

$f_x, f_u$ は以下のヤコビアンであり、上式の2項$目(( f_x ^T \lambda) ^T \delta x)$、3項目$((f_u ^T \lambda) ^T \delta u)$は2次形式となっており、スカラー値である。

$$
f_x = \frac{\partial f}{\partial x} =
\begin{bmatrix}
\dfrac{\partial f_1}{\partial x_1} & \cdots & \dfrac{\partial f_1}{ \partial x_n} \\
\vdots & \ddots \\
\dfrac{\partial f_n}{\partial x_1} & \cdots & \dfrac{\partial f_n}{\partial x_n} \\
\end{bmatrix}
, \space
f_u = \frac{\partial f}{\partial u} =
\begin{bmatrix}
\dfrac{\partial f_1}{\partial u_1} & \cdots & \dfrac{\partial f_1}{\partial u_n} \\
\vdots & \ddots \\
\dfrac{\partial f_n}{\partial u_1} & \cdots & \dfrac{\partial f_n}{\partial u_n} \\
\end{bmatrix}
$$

##### $\lambda^T \dot{x}$ の項の変分計算

$$
- \delta \left( \lambda^T \dot{x} \right) = - \left. \frac{d}{d \varepsilon} \left[ \left( \lambda+\varepsilon\delta\lambda \right)^T \left( \dot{x}+\varepsilon\delta\dot{x} \right) \right] \right|_{\varepsilon = 0}
$$

$\left( \dot{x}+\varepsilon\delta\dot{x} \right)$ は $\varepsilon$ の1次形式であり、チェインルールは不要である。<br>
積の微分より

$$
\begin{aligned}
& - \frac{d}{d \varepsilon} \left( \lambda+\varepsilon \delta\lambda \right)^T \left( \dot{x}+\varepsilon\delta\dot{x} \right) \\
=& - \left[\delta\lambda ^T \left( \dot{x}+\varepsilon\delta\dot{x} \right) + \left( \lambda+\varepsilon \delta\lambda \right)^T \delta\dot{x} \right] 
\end{aligned}
$$

さらに $\varepsilon=0$ を代入し、

$$
- \delta (\lambda^T \dot{x}) = - \left[\delta\lambda ^T \dot{x} + \lambda^T \delta\dot{x} \right] 
$$

#### $A(\varepsilon, t)$ の変分計算結果

以上の結果より

$$
\begin{aligned}
\left. \frac{d}{d \varepsilon}  A(\varepsilon, t)  \right|_{\varepsilon = 0} =  & {L_x}^T \delta x + {L_u}^T \delta u \\ 
& \quad +  \delta \lambda ^T  f  + ( f_x ^T \lambda) ^T \delta x + (f_u ^T \lambda) ^T \delta u \\ 
& \qquad - \left(\delta\lambda ^T \dot{x} + \lambda^T \delta\dot{x} \right) 
\end{aligned}
$$


$\delta \lambda, \delta x, \delta u , \delta \dot{x}$ の項で整理すると

$$
\begin{aligned}
\left. \frac{d}{d \varepsilon}  A(\varepsilon, t)  \right|_{\varepsilon = 0} = &\delta\lambda ^T(f - \dot{x}) \\
&+ (L_x + f_x^T \lambda  )^T\delta x \\
&+ (L_u + f_u ^T \lambda )^T \delta u \\
&- \lambda^T \delta \dot{x}
\end{aligned}
$$

$\delta \lambda$ の係数は $(f-\dot{x})$となり、これは状態方程式そのものである。


### Step 7 拡大評価関数 $\bar{J}$ の変分

Step 4 から Step 6 で$\bar{J}$ の一次変化 $\delta \bar{J}$ を求めた。

$$
\begin{aligned}
\delta\bar{J}
&=
\left.
\frac{d}{d\varepsilon}
\bar{J}
\left[
x+\varepsilon\delta x,\,
u+\varepsilon\delta u,\,
\lambda+\varepsilon\delta\lambda
\right]
\right|_{\varepsilon=0} \\
= &{\Phi_x} ^T \delta x(T) + \int_0^T \left( \delta\lambda ^T(f - \dot{x}) \\
+ (L_x + f_x^T \lambda  )^T\delta x
+ (L_u + f_u ^T \lambda )^T \delta u
- \lambda^T \delta \dot{x}
\right) dt
\end{aligned}
$$


## 部分積分による変分の消去

$\delta \dot{x}$ があるため、これを部分積分で消去する。

部分積分は $(f(t)g(t))' = f(t)'g(t) + f(t)g(t)' \rightarrow \int_0^T f(t)g(t)'dt = [f(t)g(t)]_0^T - \int_0^T f(t)'g(t) dt $ であり、$f(t) = \lambda ^T, \ g(t)' = \delta \dot x$ とすると、以下のように $\delta \dot{x}$ が消え、代わりに$\dot{\lambda}$が現れる。

$$
\begin{aligned}
-\int_0^T \lambda^T \delta \dot x \ dt=& - [\lambda^T \delta x]_0^T + \int_0^T \frac{d \lambda^T}{dt} \delta x dt \\
=& \lambda(0)^T \delta x(0) - \lambda(T)^T \delta x(T) + \int_0^T \dot {\lambda}^T \delta x dt
\end{aligned}
$$

初期状態は$x(0)$で固定であるため、変分 $\delta x$ により変化させることが出来ない、つまり $\delta x(0) = 0 $である。よって、

$$
\begin{aligned}
= - \lambda(T)^T \delta x(T) + \int_0^T \dot {\lambda} \delta x dt
\end{aligned}
$$

この結果を$\delta \bar{J}$に戻し、終端コストの変分 $\delta \Phi$ と、$\delta x$ へまとめると以下のようになる。

$$
\begin{aligned}
\delta \bar{J} =& ({\Phi_x} - \lambda(T))^T \delta x(T) \\
& \quad + \int_0^T \left( \delta\lambda ^T(f - \dot{x}) 
+ (L_x + f_x^T \lambda  + \dot {\lambda })^T\delta x
+ (L_u + f_u ^T \lambda )^T \delta u
\right) dt
\end{aligned}
$$



## PMP 条件

変分法と同様に、拡大評価関数$bar{J}$が停留する条件は、その一次変化$\delta \bar{J}$がゼロになることである。

状態$x$, 入力 $u$, ラグランジュ未定乗数 $\lambda$, 終端状態$x(T)$をそれぞれ以下のように微小変化させた。

$$
\begin{array}{l}
x \rightarrow x + \varepsilon \delta x \\
u \rightarrow u + \varepsilon \delta u \\
\lambda \rightarrow  \lambda + \varepsilon \delta \lambda \\
x(T) \rightarrow x(T) + \varepsilon \delta x(T)
\end{array}
$$

$\delta x, \delta u, \delta \lambda , \delta x(T)$は任意に決めることが出来る。そのため$\delta \bar{J}=0$ とするには、$\delta x, \delta u, \delta \lambda , \delta x(T)$の係数がゼロである必要がある。

これらは次のようにまとまる。

$$
\begin{array}{lcl }
\delta \lambda & \rightarrow & f - \dot{x} = 0 \\
\delta x & \rightarrow & L_x + {f_x}^T \lambda + \dot {\lambda} = 0\\
\delta u & \rightarrow & L_u + {f_u}^T \lambda = 0 \\
\delta x(T) & \rightarrow & {\Phi_x} - \lambda(T) = 0
\end{array}
$$

そしてここからPMP条件の4式が導出される。なお、ここでも$t$は略称している。

状態方程式

$$
\boxed{
\dot{x}= f(x, u)
}
$$

随伴方程式

$$
\boxed{
\dot {\lambda} = - L_x(x,u) - {f_x(x,u)}^T \lambda
}
$$

停留条件
$$
\boxed{
L_u(x,u) + {f_u(x,u)}^T \lambda = 0
}
$$

終端条件

$$
\boxed{
\lambda(T) = {\Phi_x}(x(T))
}
$$



## Hamiltonianによる記述の簡略化

PMP条件は上記の4式である。これをHamiltonianを使い整理する。

Hamiltonian $H$ は以下の式である。$H$はスカラーである。

$$
H(x,u,\lambda) = L(x,u) + \lambda^T f(x,u)
$$

これを偏微分するが、次のことを考慮する。

$$
F(x,u,\lambda) = \lambda ^T f(x,u)
$$
とする。
変分の定義($\varepsilon$で微分した後 $\varepsilon=0$とおく)で$F(x,u,\lambda)$を計算する。

$$
\delta F = \left. \frac{d}{d \varepsilon} F(x + \varepsilon \delta x ,u + \varepsilon \delta u ,\lambda + \varepsilon \delta \lambda) \right|_{\varepsilon=0}
$$

チェインルールにより計算すると、$F(x,u,\lambda)$の変分は以下の結果になる。
$$
\delta F = \left(\frac{\partial F}{\partial x} \right)^T \delta x + \left(\frac{\partial F}{\partial u} \right)^T \delta u + \left(\frac{\partial F}{\partial \lambda} \right)^T \delta \lambda 
$$

Step 6で $\lambda^T f$ の変分計算を行った。これは次の結果であった。
$$
\delta (\lambda^T f) = \delta \lambda ^T  f  + ( f_x ^T \lambda) ^T \delta x + (f_u ^T \lambda) ^T \delta u
$$

$F(x,u,\lambda) = \lambda^T f$ であるため、以下のようになる。

$$
\begin{array}{l}
\left(\dfrac{\partial(\lambda^T f)}{\partial x}\right)^T = ( f_x ^T \lambda) ^T \\
\left(\dfrac{\partial(\lambda^T f)}{\partial u}\right)^T = ( f_u ^T \lambda) ^T \\
\left(\dfrac{\partial(\lambda^T f)}{\partial \lambda}\right)^T = (f) ^T \\
\end{array}
$$

$H$を$x,u,\lambda$で偏微分し、上記の結果を当てはめると、

$$
\begin{aligned}
\frac{\partial H}{\partial x} = \frac{\partial L}{\partial x} + \frac{\partial \left( \lambda^T f \right) }{\partial x} \space \rightarrow & \space  H_x = L_x + f_x^T \lambda \\
\frac{\partial H}{\partial u} = \frac{\partial L}{\partial u} + \frac{\partial \left( \lambda^T f \right) }{\partial u} \space \rightarrow & \space H_x = L_u + f_u^T \lambda \\
\frac{\partial H}{\partial \lambda} = \frac{\partial L}{\partial \lambda} + \frac{\partial \left( \lambda^T f \right) }{\partial \lambda} \space \rightarrow & \space H_\lambda = f \\
\end{aligned}
$$



これより、PMP条件は$H$を用いて以下のように記述することが出来る。

$$
\begin{array}{l}
\dot{x} = H_\lambda \\
\dot{\lambda} = - H_x \\
H_u = 0 \\
\lambda(T) = \Phi_x(x(T)) \\
\end{array}
$$ 

## まとめ

ここまでで、最適制御問題は「最適化問題」から「PMPが与える連立方程式を解く問題」へ変換された。

「最適化問題」

$$
\min\limits_{x(t), u(t)} \space J = \Phi(x(T)) + \int_0^T L (x,u,t) dt \\
\text{subject to} \space \dot{x} = f(x,u,t), x(0) = x_0
$$

「PMPが与える連立方程式を解く問題」


$$
\begin{array}{l}
\dot{x} = H_\lambda \\
\dot{\lambda} = - H_x \\
H_u = 0 \\
\lambda(T) = \Phi_x(x(T)) \\
\end{array}
$$ 

## C/GMRESへのつながり

PMPにより、状態方程式制約付き制御問題は次の必要条件を満たす問題へ変換される。

$$
\begin{array}{l}
\dot{x} = H_\lambda \\
\dot{\lambda} = - H_x \\
H_u = 0 \\
\lambda(T) = \Phi_x(x(T)) \\
\end{array}
$$ 

これらは、状態変数 $x$ , 随伴変数 $\lambda$ , 制御入力 $u$ が同時に満たすべき連立方程式である。<br>
しかし、これらの方程式は互いに強く結合しており、一般的には解析的に解くことは難しい。

例えば、
- $x(t)$ は初期値から時間方向へ積分する
- $\lambda(t)$ は終端条件から逆時間方向へ積分する
- u$(t)$ は各時刻で $H_u=0$ を満たすように決定する。

このような境界値問題(Boundary Value Problem)となるため、数値的に解く必要がある。

### C/GMRES との関係

C/GMRESでは、このPMPから得られる必要条件を満たす制御入力列をリアルタイムに数値計算する。

具体的には

1. PMPの必要条件から非線形方程式
$$
F(U, x, t) = 0
$$
を構成する。
1. この方程式を時間変化に合わせて追従するように微分する。
1. 微分方程式をGMRESで高速に解き、制御入力を更新する。

このように、PMPはC/GMRESの出発点となる理論であり、C/GMRESはその必要条件をリアルタイムに解くための数値解法である。

本ノートでは、運動方程式による等式制約のみを扱った。実際のロボット制御では入力制限や状態制約などの不等式制約も重要であり、これらはバリア関数や内点法、制約付きPMPとして拡張される。

以上

#### 